# OwnAI — train the mini-GPT on a free Colab GPU

This notebook trains the **from-scratch** OwnAI model on Colab's free GPU. No deep-learning framework is used: the same NumPy code runs on GPU because `ownai.backend` swaps NumPy for **CuPy** automatically when a GPU is present.

**Runtime → Change runtime type → GPU** before running.

Steps: clone the repo → install → (optionally) install CuPy → ingest & index → train tokenizer → train model → plot loss → download the trained artifacts back to your machine.

In [ ]:
# 1. Get the code.
REPO_URL = "https://github.com/Zlarien/OwnAI.git"
!git clone $REPO_URL ownai_repo || echo "(already cloned)"
%cd ownai_repo

In [ ]:
# 2. Install the package (numpy, pyyaml, ...).
!pip install -q -e .

In [ ]:
# 3. Enable the GPU backend. Colab ships CuPy preinstalled; this just confirms it.
#    If missing: !pip install -q cupy-cuda12x
from ownai.backend import device_name
print("Compute device:", device_name())  # should print 'cuda' on a GPU runtime

In [ ]:
# 4. Build the corpus and retrieval index for the chosen domain.
DOMAIN = "domains/mario-wii.yaml"
!python -m ownai.cli ingest --domain $DOMAIN
!python -m ownai.cli index  --domain $DOMAIN
!python -m ownai.cli eval   --domain $DOMAIN

In [ ]:
# 5. Train the BPE tokenizer, then the mini-GPT (bump --steps for a real run).
!python -m ownai.cli tokenizer --domain $DOMAIN
!python -m ownai.cli train     --domain $DOMAIN --steps 5000 --log-every 100

In [ ]:
# 6. Plot the training loss curve.
import json, matplotlib.pyplot as plt
h = json.load(open('artifacts/mario-wii/train_history.json'))
plt.figure(figsize=(8,4)); plt.plot(h['step'], h['loss'])
plt.xlabel('step'); plt.ylabel('loss'); plt.title('OwnAI mini-GPT training loss'); plt.grid(True); plt.show()

In [ ]:
# 7. Download the trained artifacts (model + tokenizer + index) to your machine.
import shutil
from google.colab import files
shutil.make_archive('mario-wii-artifacts', 'zip', 'artifacts/mario-wii')
files.download('mario-wii-artifacts.zip')